# Deduping a messy product catalog with `matchr`

Real-world product catalogs are messy. The same item shows up with
typos, reordered words, different casing, diacritics, missing
spaces, and dropped punctuation. This notebook walks through the
common shapes of that mess and uses [`matchr`](https://github.com/mommo-codes/matchr)
to find duplicates against a clean reference catalog.

## What you'll see

- `best_match` resolving messy user queries against a 10k catalog
- `token_sort_ratio` handling reordered words (`"Oat Drink Oatly"` ↔ `"Oatly Oat Drink"`)
- Diacritic folding (`"Crème Brûlée"` ↔ `"Creme Brulee"`)
- `batch_best_match` matching hundreds of queries against the full catalog, in parallel via rayon

## Install

```bash
pip install matchr
```

In [ ]:
import random
import time

import matchr

random.seed(42)

## Build a synthetic catalog

We'll generate 10,000 unique products that look like real grocery
SKUs: `<brand> <product> <variant> <size>`. This is the *reference*
catalog — the thing we want to match incoming messy strings against.

In [ ]:
BRANDS = [
    "Oatly", "Felix", "Coca Cola", "Lay's", "Kellogg's", "Nestlé",
    "Heinz", "Marabou", "Pringles", "Lipton", "Pepsi", "Cadbury",
    "Ben & Jerry's", "Häagen-Dazs", "Doritos", "Snickers",
]
PRODUCTS = [
    "Oat Drink", "Cat Food", "Chips", "Corn Flakes", "Chocolate",
    "Ketchup", "Cookies", "Ice Cream", "Tea", "Coffee", "Soda",
    "Crackers", "Pasta", "Yogurt", "Cereal", "Granola",
]
VARIANTS = [
    "Original", "Barista", "Chocolate", "Vanilla", "Salted", "BBQ",
    "Mint", "Strawberry", "Honey", "Sour Cream", "Paprika", "Light",
    "",
]
SIZES = [
    "100g", "200g", "400g", "500g", "750g", "1kg",
    "33cl", "50cl", "1L", "1.5L", "175g", "95g", "460g",
]

def synth():
    parts = [random.choice(BRANDS), random.choice(PRODUCTS)]
    v = random.choice(VARIANTS)
    if v:
        parts.append(v)
    parts.append(random.choice(SIZES))
    return " ".join(parts)

catalog = list({synth() for _ in range(20_000)})[:10_000]
print(f"Catalog size: {len(catalog)}")
print("Samples:")
for item in catalog[:5]:
    print(f"  {item}")

## Messy real-world queries

Users don't type the canonical name. They typo, drop spaces, lose
diacritics, and reorder words. We pick a few catalog items at
random and roughen them up:

In [ ]:
def roughen(s: str) -> str:
    """Return a typoed, lowercased, possibly word-reordered version of `s`."""
    s = s.lower()
    # 30% chance: drop a random character (typo)
    if random.random() < 0.3 and len(s) > 5:
        i = random.randint(0, len(s) - 1)
        s = s[:i] + s[i + 1:]
    # 30% chance: reorder words
    if random.random() < 0.3:
        tokens = s.split()
        random.shuffle(tokens)
        s = " ".join(tokens)
    return s

truth = random.sample(catalog, 5)
queries = [roughen(t) for t in truth]

for q, t in zip(queries, truth):
    match = matchr.best_match(q, catalog, threshold=0.6)
    found = match[0] if match else None
    score = f"{match[1]:.3f}" if match else "—"
    correct = "✅" if found == t else "❌"
    print(f"{correct}  query: {q!r}")
    print(f"    truth: {t!r}")
    print(f"    found: {found!r}  (score {score})")
    print()

## Word-order tolerance: `token_sort_ratio`

When word order varies a lot, the plain combined score punishes
the comparison. `token_sort_ratio` sorts tokens before comparing,
so order doesn't matter.

In [ ]:
pairs = [
    ("Oat Drink Oatly 1L", "Oatly Oat Drink 1L"),
    ("175g Lay's Salted Chips", "Lay's Salted Chips 175g"),
    ("Felix Cat Food 400g Original", "Original Felix 400g Cat Food"),
]

print(f"{'pair':70s}  combined   token_sort   token_set")
for a, b in pairs:
    label = f"{a!r} vs {b!r}"
    print(f"{label:70s}  {matchr.combined_score(a, b):.3f}      {matchr.token_sort_ratio(a, b):.3f}        {matchr.token_set_ratio(a, b):.3f}")

## Diacritic folding

`matchr` strips diacritics during normalisation, so cross-locale
and ASCII-only data sources match.

In [ ]:
diacritic_pairs = [
    ("Crème Brûlée", "Creme Brulee"),
    ("Häagen-Dazs Vanilla", "Haagen Dazs Vanilla"),
    ("Marabou Mjölkchoklad", "Marabou Mjolkchoklad"),
]

for a, b in diacritic_pairs:
    score = matchr.combined_score(a, b)
    print(f"{a!r:30s} vs {b!r:30s}  →  {score:.3f}")

## Batch matching at scale

`batch_best_match` pre-normalises the catalog once and runs queries
in parallel across CPU cores via rayon. This is where the big
speedup vs a Python loop shows up.

Let's match 500 messy queries against the full 10k catalog and
time it.

In [ ]:
n_queries = 500
truth_batch = random.choices(catalog, k=n_queries)
query_batch = [roughen(t) for t in truth_batch]

start = time.perf_counter()
results = matchr.batch_best_match(query_batch, catalog, threshold=0.5)
elapsed = time.perf_counter() - start

correct = sum(
    1 for r, t in zip(results, truth_batch)
    if r is not None and r[0] == t
)

print(f"Matched {n_queries} queries against {len(catalog)} candidates")
print(f"  time:     {elapsed * 1000:.1f} ms ({elapsed * 1e6 / n_queries:.1f} µs/query)")
print(f"  accuracy: {correct}/{n_queries} ({100 * correct / n_queries:.1f}%)")

## Takeaways

- For one-off matching, `best_match(query, candidates, threshold)` is enough.
- For lots of queries against a fixed catalog, `batch_best_match` is what you want — it amortises catalog normalisation and parallelises across cores.
- For product/title data specifically, reach for `token_sort_ratio` so word order doesn't hurt you.
- All scorers fold diacritics by default, so multi-locale data lines up.

Source: https://github.com/mommo-codes/matchr